In [1]:
!pip install -U langgraph langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.6 MB/s eta 0:00:00


In [3]:
# LangGraph + Groq Implementation

import os
from typing import Annotated, TypedDict

from google.colab import userdata

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    SystemMessage,
    AIMessage
)

from langchain_core.tools import tool

from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


# ---------------------------------------------------------------------------
# 0. API KEY & GROQ MODEL SETUP
# ---------------------------------------------------------------------------

try:
    os.environ["GROQ_API_KEY"] = userdata.get("aaditi21")
except Exception as e:
    raise RuntimeError(
        "Please add 'Groq_API_Key' to your Colab Secrets "
        "and enable notebook access."
    ) from e


# Create Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=os.environ["GROQ_API_KEY"]
)


# ---------------------------------------------------------------------------
# 1. STATE DEFINITION
# ---------------------------------------------------------------------------

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    next_node: str


# ---------------------------------------------------------------------------
# 2. TOOL DEFINITION
# ---------------------------------------------------------------------------

@tool
def process_refund(user_id: str, amount: float) -> str:
    """Executes a financial refund for a specific user ID."""

    return (
        f"SUCCESS: Refund of ${amount} has been "
        f"processed for User '{user_id}'."
    )


# ---------------------------------------------------------------------------
# 3. SUPERVISOR ROUTER AGENT
# ---------------------------------------------------------------------------

def supervisor_agent(state: AgentState) -> AgentState:

    system_prompt = (
        "You are a Support Router. Analyze the user prompt.\n"
        "- If it is general technical troubleshooting, "
        "respond with 'SUPPORT'.\n"
        "- If it involves financial refunds or account modifications, "
        "respond with 'ACTION'.\n"
        "Respond ONLY with 'SUPPORT' or 'ACTION'."
    )

    messages = [
        SystemMessage(content=system_prompt)
    ] + state["messages"]

    response = llm.invoke(messages)

    decision = response.content.strip().upper()

    if "ACTION" in decision:
        next_step = "account_actions_agent"

    elif "SUPPORT" in decision:
        next_step = "tech_support_agent"

    else:
        next_step = END

    return {
        "next_node": next_step
    }


# ---------------------------------------------------------------------------
# 4. TECHNICAL SUPPORT AGENT
# ---------------------------------------------------------------------------

def tech_support_agent(state: AgentState) -> AgentState:

    system_prompt = SystemMessage(
        content=(
            "You are a helpful Technical Support Specialist. "
            "Provide concise troubleshooting guidance."
        )
    )

    messages = [
        system_prompt
    ] + state["messages"]

    response = llm.invoke(messages)

    return {
        "messages": [
            AIMessage(
                content=f"[Tech Support]: {response.content}"
            )
        ],
        "next_node": END
    }


# ---------------------------------------------------------------------------
# 5. ACCOUNT ACTION AGENT
# ---------------------------------------------------------------------------

def account_actions_agent(state: AgentState) -> AgentState:

    # Give refund tool to Groq LLM
    llm_with_tools = llm.bind_tools(
        [process_refund]
    )

    system_prompt = SystemMessage(
        content=(
            "You are an Account Manager. "
            "Use the process_refund tool to issue "
            "user refunds when requested."
        )
    )

    messages = [
        system_prompt
    ] + state["messages"]

    response = llm_with_tools.invoke(messages)

    # Check whether the LLM requested the refund tool
    if response.tool_calls:

        tool_call = response.tool_calls[0]

        tool_output = process_refund.invoke(
            tool_call["args"]
        )

        final_msg = (
            f"[Account Agent]: Executed tool. "
            f"Result: {tool_output}"
        )

    else:

        final_msg = (
            f"[Account Agent]: {response.content}"
        )

    return {
        "messages": [
            AIMessage(content=final_msg)
        ],
        "next_node": END
    }


# ---------------------------------------------------------------------------
# 6. CONSTRUCT LANGGRAPH WORKFLOW
# ---------------------------------------------------------------------------

workflow = StateGraph(AgentState)


# Add nodes
workflow.add_node(
    "supervisor",
    supervisor_agent
)

workflow.add_node(
    "tech_support_agent",
    tech_support_agent
)

workflow.add_node(
    "account_actions_agent",
    account_actions_agent
)


# START → Supervisor
workflow.add_edge(
    START,
    "supervisor"
)


# Supervisor → appropriate agent
workflow.add_conditional_edges(
    "supervisor",

    lambda state: state["next_node"],

    {
        "tech_support_agent": "tech_support_agent",
        "account_actions_agent": "account_actions_agent",
        END: END
    }
)


# Agents → END
workflow.add_edge(
    "tech_support_agent",
    END
)

workflow.add_edge(
    "account_actions_agent",
    END
)


# Compile graph
app = workflow.compile()


# ---------------------------------------------------------------------------
# 7. DEMONSTRATION FUNCTION
# ---------------------------------------------------------------------------

def run_demo(user_query: str):

    print(
        "\n================ USER QUERY ================\n"
    )

    print(user_query)

    inputs = {
        "messages": [
            HumanMessage(content=user_query)
        ]
    }

    result = app.invoke(inputs)

    print(
        "\n================ SYSTEM RESPONSE ================\n"
    )

    print(
        result["messages"][-1].content
    )


# ---------------------------------------------------------------------------
# 8. TEST CASE 1 - TECHNICAL SUPPORT
# ---------------------------------------------------------------------------

run_demo(
    "My app keeps freezing whenever I try to upload "
    "a PNG file. How can I fix this?"
)


# ---------------------------------------------------------------------------
# 9. TEST CASE 2 - REFUND
# ---------------------------------------------------------------------------

run_demo(
    "I was billed twice by mistake. Please refund "
    "$49.99 for my account 'user_9876'."
)


================ USER QUERY ================

My app keeps freezing whenever I try to upload a PNG file. How can I fix this?

================ SYSTEM RESPONSE ================

[Tech Support]: **Quick Checklist for PNG‑upload freezes**

| # | What to check | Why it matters | How to test |
|---|----------------|----------------|------------|
| 1 | **File size & dimensions** | Large images (>5 MB or >4000 px) can exhaust memory or hit upload limits. | Open the PNG in a viewer → note size. Try a smaller version (e.g., 800 × 800 px). |
| 2 | **File integrity** | Corrupted PNGs can cause the app to hang while parsing. | Open the file in another image editor (Paint, Preview, GIMP). If it fails, replace the file. |
| 3 | **App version** | Bugs fixed in newer releases often affect media uploads. | Check for updates → install the latest version. |
| 4 | **Cache / temp data** | Stale cache can corrupt the upload process. | Clear the app’s cache or temporary files (Settings → Storage → Clear cac